# Background Segmentation Experiments and Result Analysis for VTON Pipeline

## Sapiens

In [1]:
import os
import sys
import cv2
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

sapiens_dir = "../Sapiens-Pytorch-Inference"

if not os.path.isdir(sapiens_dir):
    raise FileNotFoundError(f"Sapiens-Ordner nicht gefunden: {sapiens_dir}")

sys.path.insert(0, sapiens_dir)

from sapiens_inference import (
    SapiensDepth,
    SapiensDepthType,
    SapiensSegmentation,
    SapiensSegmentationType,
    SapiensConfig,
)


In [2]:

config = SapiensConfig()
config.depth_type = SapiensDepthType.DEPTH_1B
config.segmentation_type = SapiensSegmentationType.SEGMENTATION_1B

if torch.cuda.is_available():
    config.device = "cuda"
else:
    print("cpu")
    config.device = "cpu"

orig_cwd = os.getcwd()
try:
    os.makedirs(os.path.join(sapiens_dir, "models"), exist_ok=True)
    os.chdir(sapiens_dir)
    depth_predictor = SapiensDepth(config.depth_type, config.device, config.dtype)
    seg_predictor = SapiensSegmentation(config.segmentation_type, config.device, config.dtype)
finally:
    os.chdir(orig_cwd)

print("device", config.device)


device cuda


In [3]:


def run_sapiens_on_image(img_path):
    pil = Image.open(img_path).convert("RGB")
    rgb_np = np.array(pil)
    bgr_np = cv2.cvtColor(rgb_np, cv2.COLOR_RGB2BGR)
    H, W = bgr_np.shape[:2]

    # Segmentation
    seg_logits = seg_predictor(bgr_np)
    if isinstance(seg_logits, torch.Tensor):
        seg_map = seg_logits.squeeze().cpu().numpy()
    else:
        seg_map = seg_logits

    if seg_map.shape != (H, W):
        seg_map = cv2.resize(seg_map, (W, H), interpolation=cv2.INTER_LINEAR)

    human_mask = (seg_map > 0.5).astype(np.uint8)

    return rgb_np, human_mask


In [ ]:
test_image = "../data/flo_16/real/images/florian_0000.jpg"

rgb, mask = run_sapiens_on_image(test_image)

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.title("Original")
plt.imshow(rgb)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("Human-Maske")
plt.imshow(mask, cmap="gray")
plt.axis("off")



plt.show()


In [4]:
# --- Sapiens: 2 Images -> Upper Clothing + (Upper/Lower Arms L/R) -> combine -> save final mask (single cell) ---

import sys
from pathlib import Path

import cv2
import numpy as np
import torch

# 1) Repo-Pfad setzen (anpassen falls nötig)
REPO_ROOT = Path.cwd()
SAPIENS_REPO = REPO_ROOT / "Sapiens-Pytorch-Inference"
sys.path.insert(0, str(SAPIENS_REPO))

# 2) Imports aus Sapiens
from sapiens_inference.segmentation import SapiensSegmentation, SapiensSegmentationType, classes

# 3) Zwei Input-Bilder (anpassen!)
image_path_1 = "../data/wandb/qwen/input_image_27_8821c7e3e1d5a0184761.png"
image_path_2 = "../data/wandb/qwen/output_image_27_62f3ac651741f9e06475.png"

# 4) Zielgröße (H, W)
TARGET_H, TARGET_W = 1248, 704

def load_and_resize_bgr(path: str, target_h: int, target_w: int) -> np.ndarray:
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        raise FileNotFoundError(f"Could not load image: {path}")

    h, w = img_bgr.shape[:2]
    if (h, w) != (target_h, target_w):
        # Resize Bild vor Segmentierung
        interp = cv2.INTER_AREA if (h > target_h or w > target_w) else cv2.INTER_LINEAR
        img_bgr = cv2.resize(img_bgr, (target_w, target_h), interpolation=interp)

    return img_bgr

def find_class_id_any(name_candidates, classes_list):
    """
    Find first matching class id for any candidate in list.
    Matching strategy: exact, case-insensitive exact, then contains (case-insensitive).
    """
    # exact
    for cand in name_candidates:
        if cand in classes_list:
            return classes_list.index(cand)

    # case-insensitive exact
    lower_map = {n.lower(): i for i, n in enumerate(classes_list)}
    for cand in name_candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    # contains
    for cand in name_candidates:
        cl = cand.lower()
        for i, n in enumerate(classes_list):
            if cl in n.lower():
                return i

    raise ValueError(
        f"Konnte keine der Klassen finden: {name_candidates}\n"
        f"Arm/Cloth Kandidaten: {[n for n in classes_list if ('arm' in n.lower() or 'cloth' in n.lower())]}"
    )

# 5) Estimator erstellen
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
estimator = SapiensSegmentation(
    SapiensSegmentationType.SEGMENTATION_1B,
    device=device,
    dtype=torch.float16 if device.type == "cuda" else torch.float32,
)

# 6) Zielklassen robust finden (inkl. deiner Namenskonvention)
upper_clothing_id = find_class_id_any(
    ["Upper Clothing"],
    classes
)

left_upper_arm_id = find_class_id_any(
    ["Left Upper Arm", "Upper Arm Left", "LeftUpperArm"],
    classes
)
right_upper_arm_id = find_class_id_any(
    ["Right Upper Arm", "Upper Arm Right", "RightUpperArm"],
    classes
)

left_lower_arm_id = find_class_id_any(
    ["Left Lower Arm", "Lower Arm Left", "LeftLowerArm"],
    classes
)
right_lower_arm_id = find_class_id_any(
    ["Right Lower Arm", "Lower Arm Right", "RightLowerArm"],
    classes
)

target_ids = np.array(
    [upper_clothing_id, left_upper_arm_id, right_upper_arm_id, left_lower_arm_id, right_lower_arm_id],
    dtype=np.int32
)

print("Using class IDs:")
for cid in target_ids:
    print(f"- {cid:3d} | {classes[cid]}")

def segment_and_build_mask(img_bgr: np.ndarray) -> np.ndarray:
    seg_map = estimator(img_bgr).astype(np.int32)  # (H, W)
    mask = np.isin(seg_map, target_ids).astype(np.uint8) * 255
    return mask

# 7) Bilder laden + ggf. resize auf 1248x704
img1_bgr = load_and_resize_bgr(image_path_1, TARGET_H, TARGET_W)
img2_bgr = load_and_resize_bgr(image_path_2, TARGET_H, TARGET_W)

# 8) Masken pro Bild
mask1 = segment_and_build_mask(img1_bgr)
mask2 = segment_and_build_mask(img2_bgr)

# 9) Endmaske: Union beider Masken
final_mask = cv2.bitwise_or(mask1, mask2)

# 10) Speichern (Endmaske)
out_dir = Path(image_path_1).with_suffix("").parent
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / (
    f"{Path(image_path_1).stem}__{Path(image_path_2).stem}_mask_upperclothing_upperlowerarms_1248x704___27.png"
)

ok = cv2.imwrite(str(out_path), final_mask)
if not ok:
    raise IOError(f"cv2.imwrite failed for: {out_path}")

print("Saved final mask:", out_path)
print("Final mask shape:", final_mask.shape, "dtype:", final_mask.dtype, "unique:", np.unique(final_mask))


Using class IDs:
-  22 | Upper Clothing
-  10 | Left Upper Arm
-  19 | Right Upper Arm
-   6 | Left Lower Arm
-  15 | Right Lower Arm
Segmentation inference took: 101.8137 seconds
Segmentation inference took: 99.2821 seconds
Saved final mask: ..\data\wandb\qwen\input_image_27_8821c7e3e1d5a0184761__output_image_27_62f3ac651741f9e06475_mask_upperclothing_upperlowerarms_1248x704___27.png
Final mask shape: (1248, 704) dtype: uint8 unique: [  0 255]


In [5]:
# --- Maske um ca. 50 Pixel in alle Richtungen vergrößern ---

DILATE_PX = 10  # gewünschte Ausdehnung

kernel_size = 2 * DILATE_PX + 1
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))

final_mask_dilated = cv2.dilate(final_mask, kernel)

# Speichern
out_path_dilated = out_path.with_name(out_path.stem + "_dilated50___27.png")
cv2.imwrite(str(out_path_dilated), final_mask_dilated)

print("Saved dilated mask:", out_path_dilated)


Saved dilated mask: ..\data\wandb\qwen\input_image_27_8821c7e3e1d5a0184761__output_image_27_62f3ac651741f9e06475_mask_upperclothing_upperlowerarms_1248x704___27_dilated50___27.png


## SAM 3

In [ ]:
from transformers import Sam3Processor, Sam3Model
import torch
from PIL import Image
import requests

device = "cuda" if torch.cuda.is_available() else "cpu"

model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

# Load image
image_url = "http://images.cocodataset.org/val2017/000000077595.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

# Segment using text prompt
inputs = processor(images=image, text="ear", return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Post-process results
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

print(f"Found {len(results['masks'])} objects")
# Results contain:
# - masks: Binary masks resized to original image size
# - boxes: Bounding boxes in absolute pixel coordinates (xyxy format)
# - scores: Confidence scores


model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]